In [3]:

# ============================================================
#  BASELINE MODELS
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Project directory
# ------------------------------------------------------------

cwd = os.getcwd()

PROJECT_ROOT = (
    os.path.dirname(cwd)
    if os.path.basename(cwd).lower() == "notebooks"
    else cwd
)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Load forecasting framework dataset
# ------------------------------------------------------------

possible_files = [
    os.path.join(
        OUTPUT_DIR,
        "forecasting_framework_dataset.csv"
    ),
    os.path.join(
        PROCESSED_DIR,
        "forecasting_framework_dataset.csv"
    )
]

source_file = next(
    (f for f in possible_files if os.path.exists(f)),
    None
)

if source_file is None:
    raise FileNotFoundError(
        "Phase 8 forecasting_framework_dataset.csv not found. "
        "Run Phase 8 successfully first."
    )

df = pd.read_csv(source_file)

print("Loaded dataset:", source_file)

# ------------------------------------------------------------
# Identify forecasting columns
# ------------------------------------------------------------

date_candidates = [
    "date",
    "report_date",
    "observation_date",
    "snapshot_date",
    "month",
    "year"
]

date_col = next(
    (
        c for c in date_candidates
        if c in df.columns
    ),
    None
)

if date_col is None:
    raise ValueError("Date column not found.")

df[date_col] = pd.to_datetime(
    df[date_col],
    errors="coerce"
)

if "forecast_origin" not in df.columns:
    df["forecast_origin"] = df[date_col]

target_candidates = [
    "total_children",
    "total_care_load",
    "care_load",
    "placement_demand",
    "total",
    "count",
    "value"
]

target_col = next(
    (
        c for c in target_candidates
        if c in df.columns
    ),
    None
)

if target_col is None:
    numeric_cols = df.select_dtypes(
        include=np.number
    ).columns.tolist()

    excluded = [
        c for c in numeric_cols
        if any(
            word in c.lower()
            for word in ["lag", "rolling", "forecast", "target"]
        )
    ]

    numeric_cols = [
        c for c in numeric_cols
        if c not in excluded
    ]

    if not numeric_cols:
        raise ValueError("Numeric forecasting target not found.")

    target_col = numeric_cols[0]

df[target_col] = pd.to_numeric(
    df[target_col],
    errors="coerce"
)

df = (
    df.dropna(subset=[date_col, target_col])
    .sort_values(date_col)
    .reset_index(drop=True)
)

print("Forecasting target:", target_col)

# ------------------------------------------------------------
# Define forecast horizons
# ------------------------------------------------------------

horizons = {
    "1_day": 1,
    "7_day": 7,
    "14_day": 14,
    "30_day": 30
}

# ------------------------------------------------------------
# Define validation and test origins
# ------------------------------------------------------------

split_file = os.path.join(
    OUTPUT_DIR,
    "chronological_split_summary.csv"
)

if not os.path.exists(split_file):
    raise FileNotFoundError(
        "Chronological split summary not found. Run Phase 8."
    )

split_summary = pd.read_csv(split_file)

validation_end = pd.to_datetime(
    split_summary.loc[
        split_summary["Dataset"] == "Validation",
        "End_Date"
    ].iloc[0]
)

validation_start = pd.to_datetime(
    split_summary.loc[
        split_summary["Dataset"] == "Validation",
        "Start_Date"
    ].iloc[0]
)

test_start = pd.to_datetime(
    split_summary.loc[
        split_summary["Dataset"] == "Testing",
        "Start_Date"
    ].iloc[0]
)

# Use the validation and test forecast origins.
validation_origins = df[
    df[date_col].between(validation_start, validation_end)
].copy()

test_origins = df[
    df[date_col] >= test_start
].copy()

if validation_origins.empty:
    raise ValueError("Validation observations are unavailable.")

# ------------------------------------------------------------
# Detect reporting frequency
# ------------------------------------------------------------

median_interval = df[date_col].diff().median()

if median_interval <= pd.Timedelta(days=1.5):
    seasonal_period = 7
    frequency = "Daily"

elif median_interval <= pd.Timedelta(days=8):
    seasonal_period = 52
    frequency = "Weekly"

else:
    seasonal_period = 12
    frequency = "Monthly or lower frequency"

print("Reporting frequency:", frequency)

# ------------------------------------------------------------
# Create chronological target lookup
# ------------------------------------------------------------

series = (
    df.set_index(date_col)[target_col]
    .sort_index()
)

# Only exact reporting-date matches are used.
# This avoids silently substituting the wrong actual date.

def get_actual_target(origin, steps):

    future_date = df.loc[
        df[date_col] > origin, date_col
    ].drop_duplicates().sort_values()

    if len(future_date) < steps:
        return None, None

    actual_date = future_date.iloc[steps - 1]

    actual = series.loc[actual_date]

    return actual_date, actual

# ------------------------------------------------------------
# Baseline prediction functions
# ------------------------------------------------------------

def persistence_prediction(history, horizon):
    return float(history.iloc[-1])


def seasonal_naive_prediction(history, horizon):
    period = seasonal_period

    if len(history) < period:
        return np.nan

    return float(history.iloc[-period])


def moving_average_prediction(history, window):
    if len(history) < window:
        return np.nan

    return float(history.iloc[-window:].mean())


# ------------------------------------------------------------
# Generate baseline predictions
# ------------------------------------------------------------

baseline_predictions = []

model_names = [
    "Naive_Persistence",
    "Seasonal_Naive",
    "Moving_Average_7",
    "Moving_Average_14",
    "Moving_Average_30"
]

forecast_datasets = {
    "Validation": validation_origins,
    "Testing": test_origins
}

for dataset_name, origins in forecast_datasets.items():

    for _, row in origins.iterrows():

        origin = row[date_col]

        # Strictly historical information.
        history = series.loc[series.index <= origin]

        if history.empty:
            continue

        for horizon_name, horizon_steps in horizons.items():

            actual_date, actual = get_actual_target(
                origin,
                horizon_steps
            )

            if actual_date is None:
                continue

            predictions = {
                "Naive_Persistence":
                    persistence_prediction(
                        history, horizon_steps
                    ),

                "Seasonal_Naive":
                    seasonal_naive_prediction(
                        history, horizon_steps
                    ),

                "Moving_Average_7":
                    moving_average_prediction(
                        history, 7
                    ),

                "Moving_Average_14":
                    moving_average_prediction(
                        history, 14
                    ),

                "Moving_Average_30":
                    moving_average_prediction(
                        history, 30
                    )
            }

            for model_name, prediction in predictions.items():

                if pd.isna(prediction):
                    continue

                baseline_predictions.append({
                    "Dataset": dataset_name,
                    "Forecast_Origin": origin,
                    "Actual_Target_Date": actual_date,
                    "Horizon": horizon_name,
                    "Model": model_name,
                    "Actual": float(actual),
                    "Predicted": float(prediction),
                    "Absolute_Error": abs(
                        float(actual) - float(prediction)
                    )
                })

baseline_predictions = pd.DataFrame(
    baseline_predictions
)

if baseline_predictions.empty:
    raise ValueError(
        "No baseline predictions generated. "
        "Check reporting frequency and available observations."
    )

# ------------------------------------------------------------
# Evaluate baseline performance
# ------------------------------------------------------------

def calculate_metrics(group):

    actual = group["Actual"]
    predicted = group["Predicted"]

    mae = mean_absolute_error(
        actual, predicted
    )

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    nonzero = actual != 0

    mape = (
        mean_absolute_percentage_error(
            actual[nonzero],
            predicted[nonzero]
        ) * 100
        if nonzero.any()
        else np.nan
    )

    return pd.Series({
        "Observations": len(group),
        "MAE": mae,
        "RMSE": rmse,
        "MAPE_Percent": mape,
        "Bias": np.mean(predicted - actual)
    })

validation_predictions = baseline_predictions[
    baseline_predictions["Dataset"] == "Validation"
].copy()

test_predictions = baseline_predictions[
    baseline_predictions["Dataset"] == "Testing"
].copy()

validation_metrics = (
    validation_predictions
    .groupby(["Model", "Horizon"])
    .apply(calculate_metrics)
    .reset_index()
)

test_metrics = (
    test_predictions
    .groupby(["Model", "Horizon"])
    .apply(calculate_metrics)
    .reset_index()
)

# ------------------------------------------------------------
# Identify best baseline using validation MAE
# ------------------------------------------------------------

best_models = (
    validation_metrics
    .sort_values("MAE")
    .groupby("Horizon")
    .first()
    .reset_index()
)

# ------------------------------------------------------------
# Save baseline results
# ------------------------------------------------------------

baseline_predictions.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "baseline_predictions.csv"
    ),
    index=False
)

validation_metrics.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "baseline_validation_metrics.csv"
    ),
    index=False
)

test_metrics.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "baseline_test_metrics.csv"
    ),
    index=False
)

best_models.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "best_baseline_models.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Final results
# ------------------------------------------------------------

print("\n========================================")
print(" BASELINE MODELS COMPLETED")
print("========================================")

print("\nBaseline models:", model_names)

print("\nValidation Performance:")
print(validation_metrics.to_string(index=False))

print("\nTesting Performance:")
print(test_metrics.to_string(index=False))

print("\nBest Baseline Models (Validation MAE):")
print(best_models.to_string(index=False))

print("\nTotal baseline predictions:",
      len(baseline_predictions))

print("\nSaved files:")
print("1. baseline_predictions.csv")
print("2. baseline_validation_metrics.csv")
print("3. baseline_test_metrics.csv")
print("4. best_baseline_models.csv")

print("\nAll baseline outputs saved successfully.")


Loaded dataset: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\outputs\forecasting_framework_dataset.csv
Forecasting target: children_in_hhs_care
Reporting frequency: Daily

 BASELINE MODELS COMPLETED

Baseline models: ['Naive_Persistence', 'Seasonal_Naive', 'Moving_Average_7', 'Moving_Average_14', 'Moving_Average_30']

Validation Performance:
            Model Horizon  Observations         MAE        RMSE  MAPE_Percent        Bias
Moving_Average_14  14_day         104.0  756.774038 1157.525125     27.385095  625.401786
Moving_Average_14   1_day         104.0  341.978709  500.986700     10.267477  290.719093
Moving_Average_14  30_day         104.0  999.912775 1612.812763     43.484666  833.007555
Moving_Average_14   7_day         104.0  560.797390  831.754473     18.332473  468.469093
Moving_Average_30  14_day         104.0 1069.098397 1537.786818     40.803275  932.418269
Moving_Average_30   1_day         104.0  672.402885  935.281491     22.228758  597.735577
Movi